In [109]:
import pandas as pd
import numpy as np
import pysam
import io

In [177]:
cluster_names_and_weights = """
Early_Erythroid_Cells 0.17
Late_Erythroid_Cells 0.15
Myeloid_Progenitor 0.1
Lymphoid_Progenitor 0.08
Granulocyte-Monocyte_Progenitor 0.16
Neutrophil 0.21
CD14+_Monocyte_Cells_1 0.05
CD14+_Monocyte_Cells_2 0.04
CD16+_Monocyte_Cells 0.04
Lymphoid_Progenitor 0.08
""".strip().split("\n")
sum(dict((x.split()[0], float(x.split()[1])) for x in cluster_names_and_weights).values())

1.0

In [30]:
tabixfile = pysam.TabixFile("/home/nboley/src/fragmentomics_tools/test/data/frag.bed.gz")

In [168]:
max_frag_len = 512
min_mapq = 10


# fragment_strands = fragment_strands[mask]

s = io.StringIO("\n".join((s.replace(',', '\t') for s in tabixfile.fetch('chrX', 6241588-256, 6241588+256))))
colnames = ['contig', 'start', 'stop', 'mapq1', 'mapq2', 'sam1', 'sam2', 'cigar1', 'cigar2', 'drop']
df = pd.read_table(s, names=colnames, usecols=colnames[:-1])

# filter fragments that are too long or that don't have a high enough mapq score
df = df.query("stop - start <= @max_frag_len and mapq1 >= @min_mapq and mapq2 >= @min_mapq")

# make sam1 the first read in the pair
first_in_pair_sam_flag = np.zeros(df.shape[0], dtype=int) - 1
second_in_pair_sam_flag = np.zeros(df.shape[0], dtype=int) - 1

first_in_pair_mask = (df.sam1&64 > 0)
first_in_pair_sam_flag[first_in_pair_mask] = df.sam1[first_in_pair_mask]
second_in_pair_sam_flag[~first_in_pair_mask] = df.sam1[~first_in_pair_mask]

second_in_pair_mask = (df.sam2&64 > 0)
first_in_pair_sam_flag[second_in_pair_mask] = df.sam2[second_in_pair_mask]
second_in_pair_sam_flag[~second_in_pair_mask] = df.sam2[~second_in_pair_mask]

df['sam1'] = first_in_pair_sam_flag
df['sam2'] = second_in_pair_sam_flag

df['strand'] = np.empty((df.shape[0],), dtype='U1') 
plus_strand_mask = ((first_in_pair_sam_flag&16 == 0)  & (second_in_pair_sam_flag&16 > 0))
df['strand'][plus_strand_mask] = '+'
minus_strand_mask = ((first_in_pair_sam_flag&16 > 0)  & (second_in_pair_sam_flag&16 == 0))
df['strand'][minus_strand_mask] = '-'


rfa = RegionFragmentArray(
    starts_0=(df.start - region.length),
    stops_0=(df.stop - region.length),
    region=region,
    max_frag_len=max_frag_len,
    validate_data=True,
    fragment_strands=df.strand,
    num_cpgs=None,
    num_meth_cpgs=None,
)

df

NameError: name 'RegionFragmentArray' is not defined

In [118]:
~first_in_pair_mask

0    False
1    False
2     True
3    False
Name: sam1, dtype: bool

In [89]:
df.query("stop - start <= @max_frag_len and mapq1 >= @min_mapq and mapq2 >= @min_mapq")

,contig,start,stop,mapq1,mapq2,sam1,sam2,cigar1,cigar2
0,chrX,6241445,6241623,60,40,99,147,45M,45M
1,chrX,6241447,6241632,60,40,99,147,45M,45M
2,chrX,6241577,6241777,40,60,163,83,45M,45M
3,chrX,6241588,6241753,40,60,99,147,45M,45M
